# 00 — Ortam ve Mimari Sozlesme Kontrolu
Bu notebook calisma ortamindaki bagimliliklari (Python, PyTorch, MediaPipe, RTMPose, OpenCV, Scikit-Learn)
ve PhysioLoad AI'nin uc durumlu karar mantigini dogrular.

## Mimari Sozlesme ve Arastirma Ilkeleri

Bu notebook, **PhysioLoad AI — Olcum Duyarli Secici Biyomekanik Analiz Mimarisi**'nin bir parcasidir:
1. **Once olc, sonra guvenilirligini sinayarak raporla:** Her biyomekanik cikti bagimsiz degerlendirilir.
2. **Uc durumlu karar sozlesmesi:** `ACCEPTED` (Kabul), `REJECTED` (Ret), `NOT_COMPUTABLE` (Hesaplanamaz).
3. **Ayri dogrulama hatti:** Referans eklem verileri asla model tahminine girdi yapilmaz; yalnizca hata analizi ve A0-A4 karsilastirmasinda kullanilir.
4. **Veri sizintisini onleme:** Kisi ve deneme bazli boluntuleme (ayni katilimci train/val/test arasinda karismaz).
5. **Esik secimi:** Sayisal esikler yalnizca validation bolumunde belirlenir; test kumesi karsilastirma icin saklanir.


In [1]:
import sys
from pathlib import Path

# Proje kok dizini ve ortam kurulumu
here = Path.cwd().resolve()
code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))

from physioload_env import setup_environment
env = setup_environment(root=code_root)
print("Ortam Basariyla Yuklendi:")
print(env.summary())


Ortam Basariyla Yuklendi:
context       : vscode local
user          : berk-can
processed ver : v1
device        : cpu
code root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
artifacts root: C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
data root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai


In [2]:
import importlib.util, platform
import numpy as np
import pandas as pd
import scipy
import sklearn

checks = {
    'python': platform.python_version(),
    'mediapipe': importlib.util.find_spec('mediapipe') is not None,
    'rtmlib': importlib.util.find_spec('rtmlib') is not None,
    'torch': importlib.util.find_spec('torch') is not None,
    'opencv': importlib.util.find_spec('cv2') is not None,
    'artifacts_writable': env.artifacts_root.exists(),
    'device': env.device,
}
for k, v in checks.items():
    print(f"  {k:20s}: {v}")


  python              : 3.11.3
  mediapipe           : True
  rtmlib              : True
  torch               : True
  opencv              : True
  artifacts_writable  : True
  device              : cpu


In [3]:
# Katman 5 ve 6: Uc durumlu karar sozlesmesinin dogrulanmasi
from physioload.physical_validity import PhysicalValidityContext
from physioload.quality import QualityFeatures, QualityGate, MeasurementStatus

gate = QualityGate()

# Senaryo 1: Gecerli eklem acisi, metrik olcek yok -> Aci ACCEPTED, Dis moment NOT_COMPUTABLE
feat_no_scale = QualityFeatures(
    visibility=0.95, pose_confidence=0.92, view_deviation_deg=8.0,
    geometry_penalty=0.05, scale_valid=False, load_definition_valid=True,
    sensitivity_penalty=0.10
)
context_no_scale = PhysicalValidityContext(metric_scale_valid=False)
decision_1 = gate.evaluate(
    feat_no_scale,
    requested_outputs=['joint_angle', 'external_moment'],
    physical_context=context_no_scale
)

print("Senaryo 1 Kararlari:")
for name, out in decision_1.outputs.items():
    print(f"  {name:18s} -> {out.status.value:15s} (Skor: {out.score}, Nedenler: {out.reasons})")

assert decision_1.outputs['joint_angle'].status == MeasurementStatus.ACCEPTED
assert decision_1.outputs['external_moment'].status == MeasurementStatus.NOT_COMPUTABLE
print("\nOrtam ve temel karar mekanizmasi basariyla dogrulandi.")


Senaryo 1 Kararlari:
  joint_angle        -> ACCEPTED        (Skor: 0.9084444444444444, Nedenler: ())
  external_moment    -> NOT_COMPUTABLE  (Skor: None, Nedenler: ('metric_scale_not_validated', 'load_magnitude_missing', 'load_application_point_missing', 'load_direction_missing'))

Ortam ve temel karar mekanizmasi basariyla dogrulandi.
